# EMR曲率公式における最高次係数

このNotebookは、論文 *Diffeomorphism Groups of Circle Bundles over Integral
Symplectic Manifolds* の定理3.4で用いられる有限次元計算を再現します。
すべての置換について符号付き和を取り、標準複素構造のコピーを縮約します。

研究で使用した完全なコードは
[EMR-Paper-Computation](https://github.com/egisatoshi/EMR-Paper-Computation)
で公開されています。


## 標準複素構造

$2k$ 次元実ベクトル空間上で

$$
J=\begin{pmatrix}0&I_k\\-I_k&0\end{pmatrix}.
$$

とします。このNotebookでは、対話的にすばやく計算できるよう $k=2$ を使います。
最後の節では、まったく同じ定義を使ったより大きな計算結果も示します。


In [1]:
def k := 2

def J :=
  generateTensor
    (\match as list integer with
       | [$i, #(i + k)] -> 1
       | [$i, #(i - k)] -> -1
       | _ -> 0)
    [2 * k, 2 * k]


In [2]:
J


$\begin{pmatrix} 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \\ -1 & 0 & 0 & 0 \\ 0 & -1 & 0 & 0 \\ \end{pmatrix}$

In [3]:
withSymbols [a, b, c] J_a~c . J_c~b


$\begin{pmatrix} -1 & 0 & 0 & 0 \\ 0 & -1 & 0 & 0 \\ 0 & 0 & -1 & 0 \\ 0 & 0 & 0 & -1 \\ \end{pmatrix}$

## 交代積

$S_k$ を次の積の符号付き和とします。

$$
S_k=\sum_{\sigma\in S_{2k}}\operatorname{sgn}(\sigma)
    \prod_{i=1}^{k}J_{\sigma(2i-1),\sigma(2i)}.
$$

`evenAndOddPermutations` は偶置換と奇置換を別々に生成します。


In [4]:
def S :=
  let (es, os) := evenAndOddPermutations (2 * k) in
    sum (map (\σ -> product (map (\i -> J_(σ (2 * i - 1))_(σ (2 * i))) (between 1 k))) es) -
    sum (map (\σ -> product (map (\i -> J_(σ (2 * i - 1))_(σ (2 * i))) (between 1 k))) os)


In [5]:
S


$-8$

## 曲率の係数

束のパラメータ $p$ の最高次項は、次のテンソルから構成されます。

$$
T_{abc}{}^d=-J_{bc}J_a{}^d+J_{ac}J_b{}^d
             +2J_{ab}J_c{}^d.
$$

添字 $a_0,\ldots,a_{k-1}$ は巡回的な鎖を作ります。テンソル積に続く `.` が
この鎖を縮約し、外側の畳み込みが偶置換と奇置換からの寄与を加算します。


In [6]:
def T_a_b_c~d :=
  -1 * J_b_c . J_a~d +
  J_a_c . J_b~d +
  2 * J_a_b . J_c~d

def S' :=
  withSymbols [a]
    let (es, os) := evenAndOddPermutations (2 * k) in
      (\xs -> foldl (+) (head xs) (tail xs))
        (map
          (\σ ->
            (\xs -> foldl (.) (head xs) (tail xs))
              (map (\i -> T_(σ (2 * i - 1))_(σ (2 * i))_(a_(modulo i k))~(a_(i - 1))) (between 1 k)))
          es) -
      (\xs -> foldl (+) (head xs) (tail xs))
        (map
          (\σ ->
            (\xs -> foldl (.) (head xs) (tail xs))
              (map (\i -> T_(σ (2 * i - 1))_(σ (2 * i))_(a_(modulo i k))~(a_(i - 1))) (between 1 k)))
          os)


In [7]:
S'


$192$

## より高い次元

同じプログラムから次の結果が得られます。

$$
\begin{array}{c|rr}
k&S_k&S'_k\\ \hline
2&-8&192\\
3&-48&0\\
4&384&61440
\end{array}
$$

Egison 5.1.0で $k=4$ の完全な計算には約25分かかります。対話的なNotebookを
$k=2$ にしておくことで、各セルをすばやく再実行できます。最初の定義を
`def k := 4` に変えるだけで、他のコードを変更せずに定理3.4の完全な計算を
実行できます。
